# Audit du score — Maick Dane Nkou

**Diagnostic uniquement : aucun entraînement, aucune sélection automatique et aucun remplacement du tokenizer.**

Ce notebook examine le modèle actuellement soumis (SHA-256 vérifié), sur les **24 000 lignes de validation officielles** :
- utilisation des 10 000 entrées du vocabulaire, globalement et par langue ;
- mots les plus coûteux dans leur contexte réel (ponctuation conservée) ;
- coût des espaces, fragmentation et reconstruction ;
- intervalles par bootstrap stratifié et apparié si plusieurs candidats sont disponibles.

Les anciennes alternatives ne sont pas recréées : ajoutez leurs fichiers si vous les avez conservés.
Sans elles, l'audit du modèle courant fonctionne, mais aucune comparaison avec leurs scores précédents ne peut être établie.

Les tests cachés ne sont jamais téléchargés. Aucun fichier de soumission n'est modifié.
Exécutez toutes les cellules, puis renvoyez l'archive `audit_reports.zip`.
Les fichiers générés ne doivent **pas** être ajoutés au dossier d'équipe dans une PR.

In [ ]:
%pip install -q "tokenizers==0.22.1" "datasets>=4,<5" numpy pandas
import tokenizers
assert tokenizers.__version__ == "0.22.1", "Redémarrez la session après installation si nécessaire"

## 1. Fonctions d'audit
Le checker officiel est épinglé et vérifié par SHA-256. Les compteurs par ligne sont recoupés avec ce checker.
Le bootstrap conserve les mêmes tirages pour tous les modèles et calcule les **ratios des sommes**, pas la moyenne des ratios par ligne.

Les intervalles sont conditionnels à cette validation : ils ne corrigent ni la sélection adaptative passée,
ni la dépendance entre passages d'un même article, ni un changement de distribution. Ils ne sont pas des probabilités de succès sur le test caché.

In [ ]:
"""Read-only tokenizer diagnostics. No training and no model export."""
import bisect
import hashlib
import importlib.util
import json
import math
import re
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tokenizers import Tokenizer, pre_tokenizers, __version__ as TOKENIZERS_VERSION

LANGUAGES = ('en', 'fr', 'ha', 'sw', 'yo', 'am')
TARGETS = ('ha', 'sw', 'yo', 'am')
DATASET_ID = 'Similoluwa/african-multilingual-tokenizer-challenge'
DATASET_REVISION = 'v1.0.0'
CHECKER_COMMIT = '75578f2400c39b1f8e31ce7e7104b37fbc470d11'
CHECKER_SHA = '1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688'
REFERENCE_SHA = '1519895eace8680d2752b333f5efd82f80ade21bcd6210704ec17de55dafe035'


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_checker(folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / 'official_utils.py'
    if not path.is_file() or sha256(path) != CHECKER_SHA:
        url = ('https://raw.githubusercontent.com/aims-ai-research-foundations/'
               f'airf-multilingual-tokenizer-challenge/{CHECKER_COMMIT}/starter/utils.py')
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
        if hashlib.sha256(data).hexdigest() != CHECKER_SHA:
            raise ValueError('Official checker hash mismatch')
        path.write_bytes(data)
    spec = importlib.util.spec_from_file_location('audit_official_checker', path)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    return helper


def byte_decoder():
    values = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    characters = values.copy()
    n = 0
    for value in range(256):
        if value not in values:
            values.append(value)
            characters.append(256 + n)
            n += 1
    return {chr(character): value for character, value in zip(characters, values)}


def inspect_model(path):
    path = Path(path)
    if TOKENIZERS_VERSION != '0.22.1':
        raise ValueError('Use tokenizers==0.22.1')
    tok = Tokenizer.from_file(str(path))
    payload = json.loads(path.read_text())
    if payload['model']['type'] != 'BPE' or payload.get('decoder', {}).get('type') != 'ByteLevel':
        raise ValueError('This diagnostic currently supports the lossless byte-level BPE candidates only')
    if payload.get('normalizer') is not None or payload.get('added_tokens') or payload.get('post_processor'):
        raise ValueError('Use candidates without normalization, added tokens or post-processing')
    if payload.get('truncation') or payload.get('padding') or payload['model'].get('dropout'):
        raise ValueError('Disable truncation, padding and dropout for diagnostics')
    vocab = tok.get_vocab(with_added_tokens=True)
    if len(vocab) > 10000 or path.stat().st_size > 20 * 1024 * 1024:
        raise ValueError('File/vocabulary limit exceeded')
    if not set(pre_tokenizers.ByteLevel.alphabet()) <= set(vocab):
        raise ValueError('Missing byte alphabet symbols')
    lookup = byte_decoder()
    alphabet = set(pre_tokenizers.ByteLevel.alphabet())
    catalog = []
    for token, index in sorted(vocab.items(), key=lambda x: x[1]):
        data = bytes(lookup[c] for c in token)
        try:
            text, complete = data.decode('utf-8'), True
        except UnicodeDecodeError:
            text, complete = data.decode('utf-8', errors='backslashreplace'), False
        catalog.append({'id': index, 'raw_token': token, 'display': text,
                        'byte_length': len(data), 'complete_utf8': complete,
                        'byte_alphabet_symbol': token in alphabet,
                        'space_prefix': data.startswith(b' '),
                        'contains_ethiopic': complete and any('ETHIOPIC' in unicodedata.name(c, '') for c in text),
                        'letters_and_punctuation': complete and any(c.isalpha() for c in text)
                        and any(unicodedata.category(c).startswith('P') for c in text)})
    summary = {'sha256': sha256(path), 'file_bytes': path.stat().st_size,
               'vocab_size': len(vocab), 'merges': len(payload['model']['merges']),
               'normalizer': payload['normalizer'], 'pre_tokenizer': payload['pre_tokenizer'],
               'decoder': payload['decoder'],
               'complete_utf8_entries': sum(x['complete_utf8'] for x in catalog),
               'partial_utf8_entries': sum(not x['complete_utf8'] for x in catalog),
               'space_prefixed_entries': sum(x['space_prefix'] for x in catalog),
               'ethiopic_entries': sum(x['contains_ethiopic'] for x in catalog),
               'letters_and_punctuation_entries': sum(x['letters_and_punctuation'] for x in catalog)}
    return tok, pd.DataFrame(catalog), summary


def validate_rows(rows, require_official_size=True):
    counts = Counter(language for language, _ in rows)
    if set(counts) != set(LANGUAGES):
        raise ValueError('All six languages are required')
    if require_official_size and counts != dict.fromkeys(LANGUAGES, 4000):
        raise ValueError('Expected 4,000 validation rows per language, 24,000 total')
    if any(not isinstance(text, str) or not text.split() for _, text in rows):
        raise ValueError('Invalid/empty evaluation text')
    digest = hashlib.sha256()
    for language, text in rows:
        digest.update(json.dumps([language, text], ensure_ascii=False).encode('utf-8') + b'\n')
    return digest.hexdigest()


def audit_candidate(path, rows, helper, out, name):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', name):
        raise ValueError('Candidate names must be simple file-safe labels')
    out = Path(out) / name
    if Path(path).resolve().is_relative_to(out.resolve()):
        raise ValueError('Keep models outside the report output directory')
    out.mkdir(parents=True, exist_ok=True)
    tok, catalog, structure = inspect_model(path)
    before = structure['sha256']
    # No full-corpus text is saved. Per-row numeric arrays support paired resampling.
    usage = {lang: Counter() for lang in LANGUAGES}
    words_usage = defaultdict(lambda: [0, 0, 0, 0])  # occurrences, tokens, excess, max
    numeric = {lang: [] for lang in LANGUAGES}
    overhead = {lang: Counter() for lang in LANGUAGES}
    unknown_id = helper._unknown_token_id(tok)
    for start in range(0, len(rows), 256):
        batch = rows[start:start + 256]
        encodings = tok.encode_batch([text for _, text in batch], add_special_tokens=False)
        for (lang, text), encoding in zip(batch, encodings, strict=True):
            restored = tok.decode(encoding.ids, skip_special_tokens=False)
            restored_skip = tok.decode(encoding.ids, skip_special_tokens=True)
            strict_loss = int(restored != text or restored_skip != text)
            official_loss = int(unicodedata.normalize('NFC', restored_skip).strip()
                                != unicodedata.normalize('NFC', text).strip())
            matches = list(re.finditer(r'\S+', text))
            starts = [m.start() for m in matches]
            ends = [m.end() for m in matches]
            costs = [0] * len(matches)
            for begin, end in encoding.offsets:
                if end <= begin:
                    overhead[lang]['zero_length_offset_tokens'] += 1
                    continue
                first = bisect.bisect_right(ends, begin)
                last = bisect.bisect_left(starts, end)
                if last <= first:
                    overhead[lang]['separator_only_tokens'] += 1
                else:
                    if last - first > 1:
                        overhead[lang]['cross_word_tokens'] += 1
                    for i in range(first, last):
                        costs[i] += 1
            for match, cost in zip(matches, costs):
                entry = words_usage[(lang, match.group())]
                entry[0] += 1
                entry[1] += cost
                entry[2] += max(0, cost - 1)
                entry[3] = max(entry[3], cost)
            usage[lang].update(encoding.ids)
            unknowns = 0 if unknown_id is None else encoding.ids.count(unknown_id)
            numeric[lang].append([len(encoding.ids), len(matches), unknowns, official_loss, strict_loss])
    arrays = {lang: np.asarray(values, dtype=np.int64) for lang, values in numeric.items()}
    official = helper.profile_submission(path, data=rows, repeats=1, verbose=False)
    if not official['valid']:
        raise ValueError(f'Official validation failed: {official["errors"]}')
    for lang in LANGUAGES:
        values = arrays[lang].sum(axis=0)
        if not math.isclose(values[0] / values[1], official['fertility'][lang], abs_tol=1e-12):
            raise ValueError('Per-row counters disagree with official fertility')
    strict_losses = sum(int(a[:, 4].sum()) for a in arrays.values())
    if strict_losses:
        raise ValueError(f'{name}: {strict_losses} strict reconstruction failures')
    if any(official['unknown_rate'].values()) or official['reconstruction_penalty'] != 0:
        raise ValueError('Candidate has unknown tokens or reconstruction loss')
    for lang in LANGUAGES:
        catalog[f'uses_{lang}'] = catalog.id.map(usage[lang]).fillna(0).astype(int)
    columns = [f'uses_{lang}' for lang in LANGUAGES]
    catalog['validation_uses'] = catalog[columns].sum(axis=1)
    catalog.sort_values(['validation_uses', 'id'], ascending=[False, True]).to_csv(out/'vocabulary_usage.csv', index=False)
    word_rows = [dict(language=lang, word=word, occurrences=v[0], total_tokens=v[1],
                      average_tokens=v[1]/v[0], excess_over_one_token=v[2], max_tokens=v[3])
                 for (lang, word), v in words_usage.items()]
    word_table = pd.DataFrame(word_rows)
    top_cost = word_table.sort_values(['language', 'excess_over_one_token'], ascending=[True, False]).groupby('language').head(100)
    top_cost.to_csv(out/'costly_words.csv', index=False)
    frequent = word_table[word_table.occurrences >= 10]
    frequent = frequent.sort_values(['language', 'average_tokens', 'occurrences'], ascending=[True, False, False]).groupby('language').head(100)
    frequent.to_csv(out/'frequent_fragmented_words.csv', index=False)
    per_language = []
    for lang in LANGUAGES:
        values = arrays[lang].sum(axis=0)
        per_language.append({'language':lang, 'rows':len(arrays[lang]), 'words':int(values[1]),
                             'tokens':int(values[0]), 'fertility':official['fertility'][lang],
                             'active_vocab_entries':len(usage[lang]), **dict(overhead[lang])})
    pd.DataFrame(per_language).to_csv(out/'language_diagnostics.csv', index=False)
    headroom = min((official['guardrail_budget'] - official['fertility'][lang]) /
                   official['guardrail_budget'] for lang in ('en', 'fr'))
    report = {'structure':structure, 'official':official, 'headroom':headroom,
              'strict_lossy_rows':strict_losses, 'validation_rows':len(rows),
              'unobserved_validation_entries':int((catalog.validation_uses == 0).sum()),
              'rare_validation_entries_1_to_5':int(catalog.validation_uses.between(1, 5).sum()),
              'offset_accounting':overhead,
              'warning':'Unused on validation does not mean useless in training or on unseen text. '
                        'Byte-alphabet entries must not be pruned. Word costs include attached punctuation; '
                        'cross-word tokens, if present, overlap multiple word counts.'}
    (out/'summary.json').write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
    np.savez_compressed(out/'row_counts.npz', **arrays)
    if sha256(path) != before:
        raise AssertionError('Model changed during audit')
    return report, arrays


def paired_bootstrap(arrays_by_model, repetitions=1000, seed=2026):
    """Paired, language-stratified row bootstrap of aggregate ratios, not row averages."""
    names = list(arrays_by_model)
    if not names or repetitions < 20:
        raise ValueError('Provide a reference and at least 20 repetitions')
    reference = arrays_by_model[names[0]]
    for arrays in arrays_by_model.values():
        for lang in LANGUAGES:
            if arrays[lang].shape != reference[lang].shape or not np.array_equal(arrays[lang][:,1], reference[lang][:,1]):
                raise ValueError('Candidates must use identical validation rows in identical order')
    rng = np.random.default_rng(seed)
    score_draws = {name:[] for name in names}
    margin_draws = {name:[] for name in names}
    penalty_draws = {name:[] for name in names}
    total_rows = sum(len(reference[lang]) for lang in LANGUAGES)
    for start in range(0, repetitions, 20):
        size = min(20, repetitions-start)
        totals = {name:np.zeros((size,6,4),dtype=np.float64) for name in names}
        for j, lang in enumerate(LANGUAGES):
            count = len(reference[lang])
            indices = rng.integers(0, count, size=(size,count))
            for name in names:
                totals[name][:,j,:] = arrays_by_model[name][lang][indices,:4].sum(axis=1)
        for name, summed in totals.items():
            fertility = summed[:,:,0] / summed[:,:,1]
            unknown = summed[:,:,2] / summed[:,:,1]
            base = (fertility[:,2:] + 100*unknown[:,2:]).mean(axis=1)
            budget = 1.15*fertility[:,2:].mean(axis=1)
            penalty = np.maximum(fertility[:,:2]-budget[:,None],0).sum(axis=1)
            reconstruction_penalty = 3*summed[:,:,3].sum(axis=1)/total_rows
            score_draws[name].extend(base+penalty+reconstruction_penalty)
            margin_draws[name].extend(((budget[:,None]-fertility[:,:2])/budget[:,None]).min(axis=1))
            penalty_draws[name].extend(penalty)
    baseline = np.asarray(score_draws[names[0]])
    result=[]
    for name in names:
        scores=np.asarray(score_draws[name]); margins=np.asarray(margin_draws[name])
        delta=scores-baseline
        result.append({'candidate':name, 'score_p025':float(np.quantile(scores,.025)),
                       'score_p975':float(np.quantile(scores,.975)),
                       'delta_vs_reference_p025':float(np.quantile(delta,.025)),
                       'delta_vs_reference_p975':float(np.quantile(delta,.975)),
                       'fraction_resamples_better':float(np.mean(delta<0)),
                       'headroom_p025':float(np.quantile(margins,.025)),
                       'headroom_p975':float(np.quantile(margins,.975)),
                       'fraction_resamples_guardrail_penalized':float(np.mean(np.asarray(penalty_draws[name])>0))})
    return result


def run_audit(candidate_paths, rows, helper, output_dir, *, repetitions=1000, require_official_size=True):
    data_sha=validate_rows(rows, require_official_size)
    reports, arrays={},{}
    output_dir=Path(output_dir)
    for name,path in candidate_paths.items():
        print('Auditing',name,flush=True)
        reports[name],arrays[name]=audit_candidate(path,rows,helper,output_dir,name)
    intervals=paired_bootstrap(arrays,repetitions)
    pd.DataFrame(intervals).to_csv(output_dir/'paired_bootstrap.csv',index=False)
    manifest={'dataset':DATASET_ID if require_official_size else 'TEST_FIXTURE_NOT_OFFICIAL_VALIDATION',
              'revision':DATASET_REVISION if require_official_size else None,
              'validation_rows':len(rows),'row_order_sha256':data_sha,
              'official_size_enforced':require_official_size,
              'checker_commit':CHECKER_COMMIT,'checker_sha256':CHECKER_SHA,
              'bootstrap_seed':2026,'bootstrap_repetitions':repetitions,
              'candidates':{name:report['structure']['sha256'] for name,report in reports.items()},
              'limitations':'Intervals are conditional on observed validation rows and row-level resampling. '
                            'They do not correct prior adaptive selection, document dependence or domain shift, '
                            'and are not probabilities of performance on the hidden test. '
                            'No candidate is selected, trained or exported by this audit.'}
    (output_dir/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    table=pd.DataFrame([{'candidate':name,'full_score':r['official']['score'],
                        'reconstruction':r['official']['reconstruction'],
                        'guardrail_penalty':r['official']['guardrail_penalty'],
                        'headroom':r['headroom'],'unobserved_validation_entries':r['unobserved_validation_entries']}
                       for name,r in reports.items()])
    table.to_csv(output_dir/'comparison.csv',index=False)
    print(table.to_string(index=False))
    return reports,intervals


## 2. Modèles et données
Le modèle de référence peut être présent dans votre clone ; sinon une copie de votre propre modèle est téléchargée depuis un commit immuable.

Les alternatives `train-space_word-b3` et `weights-b4` sont détectées si les fichiers de l'ancien notebook sont encore dans la session Colab.
Vous pouvez aussi renseigner `EXTRA_CANDIDATES` avec des chemins vers vos propres modèles sauvegardés.

Laissez `VALIDATION_CSV = None` pour charger uniquement le split public `validation` de la version officielle `v1.0.0`.
En cas d'import CSV, il doit provenir de ce même split, avec exactement les colonnes `language,text`, sans modifier les chaînes.

In [ ]:
from datetime import datetime, timezone

VALIDATION_CSV = None  # Exemple : "/content/validation.csv"
EXTRA_CANDIDATES = {}  # Exemple : {"b4": "/content/weights-b4/tokenizer.json"}
BOOTSTRAP_REPETITIONS = 1000

# Les résultats restent hors de submissions/, même depuis un sous-dossier du dépôt.
project = next((p / "pyproject.toml" for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "pyproject.toml").is_file()), None)
workspace = project.parent if project is not None else Path.cwd()
AUDIT_DIR = workspace / "artifacts" / ("score-audit-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
AUDIT_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR = AUDIT_DIR / "reports"

local = workspace / "submissions/maick-dane-nkou/tokenizer.json"
if local.is_file() and sha256(local) == REFERENCE_SHA:
    reference_path = local
else:
    url = ("https://raw.githubusercontent.com/maick-code/airf-multilingual-tokenizer-challenge/"
           "5f8573cac79eea5d4406edfb5a3f71afbcc47eb1/submissions/maick-dane-nkou/tokenizer.json")
    with urllib.request.urlopen(url, timeout=60) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != REFERENCE_SHA:
        raise ValueError("Empreinte du modèle de référence incorrecte")
    reference_path = AUDIT_DIR / "input/reference/tokenizer.json"
    reference_path.parent.mkdir(parents=True, exist_ok=True)
    reference_path.write_bytes(content)
assert sha256(reference_path) == REFERENCE_SHA
candidate_paths = {"reference": reference_path}
for label in ("train-space_word-b3", "weights-b4"):
    path = workspace / "artifacts/lossless-bpe-search/candidates" / label / "tokenizer.json"
    if path.is_file():
        candidate_paths[label] = path
for label, path in EXTRA_CANDIDATES.items():
    if label == "reference":
        raise ValueError("Ne remplacez pas la référence")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    candidate_paths[label] = path
print("Modèles disponibles :", list(candidate_paths))
if len(candidate_paths) == 1:
    print("Alternatives absentes : diagnostic du modèle actuel seulement, sans comparaison des anciens essais.")

if VALIDATION_CSV is None:
    from datasets import load_dataset
    dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION, split="validation")
    rows = list(zip(dataset["language"], dataset["text"], strict=True))
    data_source = {"type": "official Hugging Face validation split", "revision": DATASET_REVISION,
                   "dataset": DATASET_ID, "fingerprint": dataset._fingerprint}
else:
    frame = pd.read_csv(VALIDATION_CSV, dtype=str, keep_default_na=False, na_filter=False)
    if set(frame.columns) != {"language", "text"}:
        raise ValueError("Colonnes attendues : language,text")
    rows = list(zip(frame.language, frame.text, strict=True))
    data_source = {"type": "participant-provided CSV; source release must be confirmed by participant",
                   "file_sha256": sha256(VALIDATION_CSV)}
validate_rows(rows, require_official_size=True)
print("Validation :", len(rows), "lignes ;", dict(Counter(language for language, _ in rows)))
helper = load_checker(AUDIT_DIR / "helper")

## 3. Lancer l'audit
L'audit encode les textes, mais n'entraîne et ne sauvegarde aucun modèle.
Un échec de reconstruction ou une incohérence avec le checker interrompt le diagnostic du candidat concerné.
Les scores des modèles non présents ne sont pas inventés à partir des anciens logs.

In [ ]:
reports, intervals = run_audit(candidate_paths, rows, helper, REPORT_DIR,
                               repetitions=BOOTSTRAP_REPETITIONS, require_official_size=True)
manifest_path = REPORT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text())
manifest["data_source"] = data_source
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("\nBootstrap conditionnel à la validation :")
print(pd.DataFrame(intervals).to_string(index=False))
for label, original in candidate_paths.items():
    assert sha256(original) == reports[label]["structure"]["sha256"]
print("\nLes fichiers des modèles sont inchangés.")

## 4. Rapports à partager

- `comparison.csv` : score complet, reconstruction, marge EN/FR.
- `paired_bootstrap.csv` : intervalles du score, de la marge et de l'écart apparié à la référence.
- `<candidat>/vocabulary_usage.csv` : fréquence de chaque token dans la validation.
- `<candidat>/costly_words.csv` : 100 mots les plus coûteux par langue, selon les tokens supplémentaires au-delà d'un token par occurrence.
- `<candidat>/frequent_fragmented_words.csv` : mots apparaissant au moins 10 fois, classés par fragmentation moyenne.
- `<candidat>/language_diagnostics.csv` : nombre de mots, tokens, entrées actives et coût des séparateurs.
- `<candidat>/row_counts.npz` : compteurs numériques par ligne pour vérifier les comparaisons.

**Important :** une entrée non observée dans la validation n'est pas forcément inutile. Ne supprimez pas les symboles de l'alphabet byte-level.
Les fragments UTF-8 incomplets sont normaux pour un tokenizer byte-level.

L'archive contient des mots issus du jeu public et des compteurs, pas le corpus complet.
Transmettez-la pour interprétation **avant** de décider de nouveaux entraînements ou d'un remplacement du modèle.

In [ ]:
import shutil
archive = shutil.make_archive(str(AUDIT_DIR / "audit_reports"), "zip", root_dir=REPORT_DIR)
print("Archive :", archive)
try:
    from google.colab import files
except ImportError:
    print("Récupérez l'archive sur le disque.")
else:
    files.download(archive)